# README example

#### As at **2026/06/03**

In [2]:
from valimp import parse, Parser, Coerce
from typing import Annotated, Union, Optional, Any

@parse  # add the `valimp.parse`` decorator to a public function or method
def public_function(
    # validate against built-in or custom types
    a: str,
    /,  # support for positional-only arguments
    # support for type unions
    b: int | float,  # or Python 3.9 `Union[int, float]`
    # validate type of container items
    c: dict[str, int | float],  # dict[str, Union[int, float]]
    # coerce input to a specific type
    d: Annotated[
        int | float | str,  # Union[int, float, str]
        Coerce(int)
    ],
    # parse input with reference to earlier inputs...
    e: Annotated[
        str,
        Parser(lambda name, obj, params: obj + f"_{name}_{params['a']}")
    ],
    # coerce and parse input...
    f: Annotated[
        str | int,  # Union[str, int]
        Coerce(str),
        Parser(lambda name, obj, _: obj + f"_{name}")
    ],
    # validate input is a class (rather than an instance)
    g: type,
    # validate input is subclass of a specific class (or that class itself) ...
    h: type[int],
    # ... or of specific classes...
    i: type[int | str],  #  type[Union[int, str]]
    # support for packing extra arguments if required, can be optionally typed...
    *args: Annotated[
        int | float | str,  # Union[int, float, str]
        Coerce(int)
    ],
    # support for optional types
    j: str | None,  # Optional[str]
    # define default values dynamically with reference to earlier inputs
    k: Annotated[
        float | None,  # Optional[float]
        Parser(lambda _, obj, params: params["b"] if obj is None else obj)
    ] = None,
    # support for packing excess kwargs if required, can be optionally typed...
    # **kwargs: int | float  # Union[int, float]
) -> dict[str, Any]:
    return {"a":a, "b":b, "c":c, "d":d, "e":e, "f":f, "g":g, "h":h, "i":i, "args":args, "j":j, "k":k}

public_function(
    # NB 'a' must be passed positionally, 'b' through 'i' can be passed positionally
    "zero",  # a
    1.0,  # b
    {"two": 2},  # c
    3.3,  # d, will be coerced from float to int, i.e. to 3
    "four",  # e, will be parsed to "four_e_zero"
    5,  # f, will be coerced to str and then parsed to "5_f"
    str,  # g
    bool,  # h, a subclass of int
    int,  # i, one of the subscripted classes
    "10",  # extra arg, will be coerced to int and packed
    20,  # extra arg, will be packed
    j="keyword_arg_j",
    # k, not passed, will be assigned dynamically as parameter b (i.e. 1.0)
)

{'a': 'zero',
 'b': 1.0,
 'c': {'two': 2},
 'd': 3,
 'e': 'four_e_zero',
 'f': '5_f',
 'g': str,
 'h': bool,
 'i': int,
 'args': (10, 20),
 'j': 'keyword_arg_j',
 'k': 1.0}

In [ ]:
public_function(
    ["not a string"],  # INVALID
    b="not an int or a float",  # INVALID
    c={2: "two"},  # INVALID, key not a str and value not an int or float
    d=3.2, # valid input
    e="valid input",
    f=5.0,  # INVALID, not a str or an int
    g=str,  # valid input
    h=str,  # INVALID, str is not int or a subclass of int
    i=bool,  # valid input
    j="valid input",
)

```
InputsError: The following inputs to 'public_function' do not conform with the corresponding type annotation:

a
	Takes type <class 'str'> although received '['not a string']' of type <class 'list'>.

b
	Takes input that conforms with <(<class 'int'>, <class 'float'>)> although received 'not an int or a float' of type <class 'str'>.

c
	Takes type <class 'dict'> with keys that conform to the first argument and values that conform to the second argument of <dict[str, int | float]>, although the received dictionary contains an item with key '2' of type <class 'int'> and value 'two' of type <class 'str'>.

f
	Takes input that conforms with <(<class 'str'>, <class 'int'>)> although received '5.0' of type <class 'float'>.

h
	Takes a subclass of <class 'int'> although received '<class 'str'>'.
```

In [ ]:
public_function(
    "zero",
    "invalid input",  # invalid (not int or float), included in errors
    {"two": 2},
    3.2,
    # no argument passed for required positional args 'e', 'f', 'g', 'h' and 'i'
    a="a again",  # 'a' is positional-only: cannot be passed as a kwarg unless sig has **kwargs
    c={"three": 3},  # passing multiple values for 'c'
    not_a_kwarg="not a kwarg",  # including an unexpected kwarg
    # no argument passed for required keyword arg 'j'
)

```
InputsError: Inputs to 'public_function' do not conform with the function signature:

Got multiple values for argument: 'c'.

Got unexpected keyword argument: 'not_a_kwarg'.

Got positional-only argument as keyword argument (and signature makes no provision for **kwargs that would otherwise receive it): 'a'.

Missing 5 positional arguments: 'e', 'f', 'g', 'h' and 'i'.

Missing 1 keyword-only argument: 'j'.

The following inputs to 'public_function' do not conform with the corresponding type annotation:

b
	Takes input that conforms with <(<class 'int'>, <class 'float'>)> although received 'invalid input' of type <class 'str'>.
    ```

In [6]:
from valimp import parse_cls
import dataclasses

@parse_cls  # place valimp decorator above the dataclass decorator
@dataclasses.dataclass
class ADataclass:
    
    a: str
    b: Annotated[
        str | int,  # Union[str, int]
        Coerce(str),
        Parser(lambda name, obj, params: obj + f" {name} {params['a']}")
    ]

rtrn = ADataclass("I'm a and will appear at the end of b", 33)
dataclasses.asdict(rtrn)

{'a': "I'm a and will appear at the end of b",
 'b': "33 b I'm a and will appear at the end of b"}